# Visual Perception project test (Colab + TensorRT)

Run this notebook with **Runtime → Change runtime type → GPU**. It loads the project from `MyDrive/visual_perception`, exports the detector and pose model to TensorRT engines when needed, runs `run_pipeline.py`, validates the JSONL, and copies the result to Drive.

Before running, confirm the input video and `.pt` weights paths in the settings cell. TensorRT engines are specific to the GPU/runtime; this notebook caches them using a key that includes the GPU, TensorRT/Ultralytics versions, weights, and export settings.

In [ ]:
# GPU and Colab runtime check
import sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is active. In Colab choose Runtime > Change runtime type > GPU, then reconnect and rerun.')
!nvidia-smi

In [ ]:
# Mount Drive and set paths. Adjust these values to match your files.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

PROJECT_DRIVE = Path('/content/drive/MyDrive/visual_perception')
INPUT_DRIVE = PROJECT_DRIVE / 'work' / 'input_video.mp4'  # change filename, or use a photo
DETECTOR_PT_DRIVE = PROJECT_DRIVE / 'models' / 'detection' / 'yolo26s.pt'
POSE_PT_DRIVE = PROJECT_DRIVE / 'models' / 'pose' / 'yolo26s-pose.pt'
DRIVE_OUTPUT = PROJECT_DRIVE / 'outputs' / 'colab_tensorrt'

# Supply the actual capture start epoch in seconds for a video.
# For a photo/test clip without capture metadata, use a documented test value.
CAMERA_ID = 'camera_01'
CAPTURE_START_EPOCH = 0.0  # replace with actual capture time when known

# Inference settings
IMGSZ = 640
BATCH_SIZE = 4
DETECTOR_CONF = 0.10
POSE_CONF = 0.25
NMS_IOU = 0.70
MAX_DETECTIONS = 300
PERSON_TRACKER = PROJECT_DRIVE / 'configs' / 'trackers' / 'bytetrack_buffer_60.yaml'
OBJECT_TRACKER = 'bytetrack.yaml'
MAX_FRAMES = 600  # set to None to process the full input

for p in [PROJECT_DRIVE / 'run_pipeline.py', INPUT_DRIVE, DETECTOR_PT_DRIVE, POSE_PT_DRIVE]:
    print(('FOUND ' if p.exists() else 'MISSING'), p)
if not (PROJECT_DRIVE / 'run_pipeline.py').is_file():
    raise FileNotFoundError('Drive project must contain run_pipeline.py at its root.')
for p in [INPUT_DRIVE, DETECTOR_PT_DRIVE, POSE_PT_DRIVE]:
    if not p.is_file():
        raise FileNotFoundError(f'Update the path in this cell; file not found: {p}')

In [ ]:
# Install project dependencies and the TensorRT wheel matching PyTorch CUDA.
# This is needed once per fresh Colab runtime.
import subprocess, re
def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

# Keep Colab's existing CUDA-enabled torch; skip torch pins in the project requirements.
req_file = PROJECT_DRIVE / 'requirements.txt'
if req_file.exists():
    reqs = []
    for line in req_file.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        package = re.split(r'[<>=!~; ]', line, maxsplit=1)[0].lower().replace('_', '-')
        if package not in {'torch', 'torchvision', 'torchaudio', 'tensorrt'}:
            reqs.append(line)
    if reqs:
        pip_install(reqs)
else:
    print('requirements.txt not found; installing runtime basics.')
pip_install(['ultralytics', 'opencv-python-headless', 'pyyaml'])

cuda_major = torch.version.cuda.split('.')[0]
if cuda_major not in {'12', '13'}:
    raise RuntimeError(f'PyTorch CUDA {torch.version.cuda} is not CUDA 12.x/13.x; select a compatible Colab GPU runtime.')
trt_package = f'tensorrt-cu{cuda_major}'
print('Installing', trt_package, 'for CUDA', torch.version.cuda)
pip_install([trt_package])
import tensorrt as trt
logger = trt.Logger(trt.Logger.ERROR)
if trt.Builder(logger) is None:
    raise RuntimeError('TensorRT loaded but could not create a builder.')
import ultralytics
print('TensorRT:', trt.__version__, '| Ultralytics:', ultralytics.__version__)

In [ ]:
# Copy only the runnable project code/configuration, media, and weights to Colab's local disk.
# Running from /content avoids slow per-frame reads from mounted Drive.
import shutil
PROJECT_LOCAL = Path('/content/visual_perception')
if PROJECT_LOCAL.exists():
    shutil.rmtree(PROJECT_LOCAL)
PROJECT_LOCAL.mkdir(parents=True)
for name in ['run_pipeline.py', 'predict_media.py', 'requirements.txt', 'pyproject.toml']:
    src = PROJECT_DRIVE / name
    if src.is_file(): shutil.copy2(src, PROJECT_LOCAL / name)
for name in ['src', 'configs', 'schemas', 'tools', 'models']:
    src = PROJECT_DRIVE / name
    if src.is_dir(): shutil.copytree(src, PROJECT_LOCAL / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
LOCAL_INPUT = PROJECT_LOCAL / 'work' / INPUT_DRIVE.name
LOCAL_INPUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(INPUT_DRIVE, LOCAL_INPUT)
LOCAL_DETECTOR_PT = PROJECT_LOCAL / DETECTOR_PT_DRIVE.relative_to(PROJECT_DRIVE)
LOCAL_POSE_PT = PROJECT_LOCAL / POSE_PT_DRIVE.relative_to(PROJECT_DRIVE)
print('Local project:', PROJECT_LOCAL)
print('Input:', LOCAL_INPUT, f'({LOCAL_INPUT.stat().st_size / 1024**2:.1f} MiB)')

In [ ]:
# Build or reuse TensorRT FP16 engines. First-time export takes longer; matching cached engines are reused.
import hashlib, shutil, json, platform
from ultralytics import YOLO

TRT_CACHE_DRIVE = DRIVE_OUTPUT / 'engines'
TRT_CACHE_DRIVE.mkdir(parents=True, exist_ok=True)
GPU_NAME = torch.cuda.get_device_name(0)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def engine_for(pt_path, task):
    key_data = {'weights': file_sha256(pt_path), 'task': task, 'imgsz': IMGSZ, 'batch': BATCH_SIZE,
                'dynamic': True, 'fp16': True, 'gpu': GPU_NAME, 'tensorrt': trt.__version__,
                'ultralytics': ultralytics.__version__}
    key = hashlib.sha256(json.dumps(key_data, sort_keys=True).encode()).hexdigest()[:20]
    cache_dir = TRT_CACHE_DRIVE / f'{task}_{key}'
    cache_dir.mkdir(parents=True, exist_ok=True)
    cached = cache_dir / (Path(pt_path).stem + '.engine')
    local_engine_dir = Path('/content/trt_engines') / f'{task}_{key}'
    local_engine_dir.mkdir(parents=True, exist_ok=True)
    local_engine = local_engine_dir / cached.name
    if cached.is_file():
        shutil.copy2(cached, local_engine)
        print(f'Reusing cached {task} engine:', cached)
        return local_engine
    model = YOLO(str(pt_path), task=task)
    exported = model.export(format='engine', imgsz=IMGSZ, batch=BATCH_SIZE, dynamic=True,
                            half=True, device=0, workspace=4)
    exported = Path(exported)
    if not exported.is_file():
        raise FileNotFoundError(f'Ultralytics did not produce the expected TensorRT engine: {exported}')
    shutil.copy2(exported, cached)
    shutil.copy2(exported, local_engine)
    (cache_dir / 'build_info.json').write_text(json.dumps(key_data, indent=2))
    print(f'Cached {task} engine:', cached)
    return local_engine

DETECTOR_ENGINE = engine_for(LOCAL_DETECTOR_PT, 'detect')
POSE_ENGINE = engine_for(LOCAL_POSE_PT, 'pose')
print('Detector engine:', DETECTOR_ENGINE)
print('Pose engine:', POSE_ENGINE)

In [ ]:
# Run the full project pipeline and write one JSON object per line.
import subprocess, shlex
LOCAL_OUTPUT = PROJECT_LOCAL / 'outputs' / 'colab_tensorrt'
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, 'run_pipeline.py', str(LOCAL_INPUT),
           '--camera-id', CAMERA_ID, '--capture-start-epoch', str(CAPTURE_START_EPOCH),
           '--detector-weights', str(DETECTOR_ENGINE), '--pose-weights', str(POSE_ENGINE),
           '--output', str(LOCAL_OUTPUT), '--imgsz', str(IMGSZ),
           '--det-conf', str(DETECTOR_CONF), '--pose-conf', str(POSE_CONF),
           '--nms-iou', str(NMS_IOU), '--max-detections', str(MAX_DETECTIONS),
           '--object-tracker', OBJECT_TRACKER, '--person-tracker', str(PERSON_TRACKER),
           '--device', '0', '--batch-size', str(BATCH_SIZE), '--start-frame', '0']
if MAX_FRAMES is not None:
    command += ['--max-frames', str(MAX_FRAMES)]
print('Running:', shlex.join(command))
subprocess.run(command, cwd=PROJECT_LOCAL, check=True)
jsonl_files = sorted(LOCAL_OUTPUT.glob('*_predictions.jsonl'), key=lambda p: p.stat().st_mtime)
if not jsonl_files:
    raise FileNotFoundError(f'No *_predictions.jsonl output found in {LOCAL_OUTPUT}')
LOCAL_JSONL = jsonl_files[-1]
print('JSONL result:', LOCAL_JSONL, f'({LOCAL_JSONL.stat().st_size / 1024**2:.2f} MiB)')

In [ ]:
# Validate and copy the JSONL result to Drive.
validator = PROJECT_LOCAL / 'tools' / 'validate_jsonl.py'
if validator.is_file():
    subprocess.run([sys.executable, str(validator), str(LOCAL_JSONL)], cwd=PROJECT_LOCAL, check=True)
else:
    print('Project validator tools/validate_jsonl.py not found; skipping schema validation.')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
DRIVE_JSONL = DRIVE_OUTPUT / LOCAL_JSONL.name
shutil.copy2(LOCAL_JSONL, DRIVE_JSONL)
print('Saved validated JSONL to Drive:', DRIVE_JSONL)